# Building a Claude-Powered API with Cloudflare Workers

This notebook shows you how to deploy Claude as an edge API endpoint using Cloudflare Workers.

**What you'll build:** A smart chat API endpoint that runs at the edge — globally distributed, with no server management.

**Why Cloudflare Workers?**
- **Global low-latency:** 300+ edge locations mean your API runs close to every user
- **Generous free tier:** 100,000 requests/day at no cost
- **No server management:** No EC2, no Docker, no ops burden
- **Fast cold starts:** Workers boot in ~0ms (V8 isolates, not containers)

**Prerequisites:**
- A Cloudflare account (free at cloudflare.com)
- Wrangler CLI installed: `npm install -g wrangler`
- An Anthropic API key
- Node.js 18+

## Architecture Overview

```
Client
  │
  │  POST /chat  { messages: [...] }
  ▼
Cloudflare Edge (nearest location)
  │
  │  fetch() to Anthropic REST API
  ▼
Anthropic API  →  Claude model
  │
  │  JSON response (or SSE stream)
  ▼
Cloudflare Edge  →  Client
```

**Key constraints to know:**
| Limit | Value | Impact |
|-------|-------|--------|
| CPU time | 30s (free), 5min (paid) | Plenty for Claude calls |
| Memory | 128MB | Enough for chat history |
| Filesystem | None | Use KV/R2/D1 for storage |
| Anthropic SDK | Not usable | Must call REST API via `fetch` |

**Workers vs a backend server:**
- **Use Workers when:** Low-latency matters, traffic is unpredictable, you want zero ops, request logic is simple
- **Use a backend server when:** You need long-running processes, local filesystem access, complex stateful logic, or heavy compute

## Python Setup

The two code cells in this notebook demonstrate:
1. Calling the Anthropic API via the Python SDK (how you'd call a deployed Worker)
2. Calling the Anthropic REST API directly with `httpx` (exactly what the Worker does internally)

Install dependencies:

In [ ]:
# Install required packages
# Using uv for fast installs (pip install uv if needed)
%pip install anthropic httpx

## Project Setup

Run these commands in your terminal to scaffold a new Worker project:

```bash
# Create a new Worker project
npm create cloudflare@latest claude-chat-worker -- --type hello-world
cd claude-chat-worker

# Store your Anthropic API key as a secret (never in code)
wrangler secret put ANTHROPIC_API_KEY
# Wrangler will prompt you to paste the key
```

Your `wrangler.toml` should look like:

```toml
name = "claude-chat-worker"
main = "src/worker.js"
compatibility_date = "2024-01-01"

# KV namespace for session storage (added later)
[[kv_namespaces]]
binding = "SESSIONS"
id = "your-kv-namespace-id"
```

## The Worker Code — Basic Chat Endpoint

Workers run JavaScript (or TypeScript). They **cannot** use the Anthropic Python/Node SDK — instead they call the REST API directly using the built-in `fetch` function.

Here is a minimal but production-ready Worker:

```javascript
// src/worker.js

const ANTHROPIC_API_URL = 'https://api.anthropic.com/v1/messages';

// CORS headers — allow any origin in this example
const CORS_HEADERS = {
  'Access-Control-Allow-Origin': '*',
  'Access-Control-Allow-Methods': 'POST, OPTIONS',
  'Access-Control-Allow-Headers': 'Content-Type',
};

export default {
  async fetch(request, env) {
    // Handle CORS preflight
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: CORS_HEADERS });
    }

    if (request.method !== 'POST') {
      return new Response('Method not allowed', { status: 405 });
    }

    let body;
    try {
      body = await request.json();
    } catch {
      return new Response(JSON.stringify({ error: 'Invalid JSON' }), {
        status: 400,
        headers: { 'Content-Type': 'application/json', ...CORS_HEADERS },
      });
    }

    const { messages } = body;
    if (!messages || !Array.isArray(messages) || messages.length === 0) {
      return new Response(JSON.stringify({ error: 'messages array is required' }), {
        status: 400,
        headers: { 'Content-Type': 'application/json', ...CORS_HEADERS },
      });
    }

    // Call Anthropic REST API
    // env.ANTHROPIC_API_KEY is injected from the secret you stored with wrangler
    const anthropicResponse = await fetch(ANTHROPIC_API_URL, {
      method: 'POST',
      headers: {
        'x-api-key': env.ANTHROPIC_API_KEY,
        'anthropic-version': '2023-06-01',
        'content-type': 'application/json',
      },
      body: JSON.stringify({
        model: 'claude-haiku-4-5',
        max_tokens: 1024,
        messages,
      }),
    });

    if (!anthropicResponse.ok) {
      const error = await anthropicResponse.json();
      return new Response(JSON.stringify({ error }), {
        status: anthropicResponse.status,
        headers: { 'Content-Type': 'application/json', ...CORS_HEADERS },
      });
    }

    const data = await anthropicResponse.json();
    const reply = data.content[0].text;

    return new Response(JSON.stringify({ reply }), {
      headers: { 'Content-Type': 'application/json', ...CORS_HEADERS },
    });
  },
};
```

**Key points:**
- `env.ANTHROPIC_API_KEY` is the secret you stored with `wrangler secret put`
- Headers `x-api-key`, `anthropic-version`, and `content-type` are all required by the Anthropic API
- `claude-haiku-4-5` is the fastest and cheapest model — ideal for edge deployments
- The Worker returns just `{ reply }` — you can extend this to return the full Anthropic response object

## Python Cell 1: Calling Claude via the SDK

This is how you'd call Claude from Python — the same interface you'd use to call your deployed Worker endpoint (once it's live at `https://claude-chat-worker.your-subdomain.workers.dev`).

In [ ]:
import os

import anthropic

client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
)

# This mirrors the request your Worker will receive and forward to Anthropic
messages = [
    {"role": "user", "content": "What are the top 3 benefits of running AI at the edge?"}
]

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=messages,
)

print(response.content[0].text)

**Expected output:**
```
The top 3 benefits of running AI at the edge are:

1. **Lower latency** — Processing happens close to the user, reducing round-trip time from hundreds of milliseconds to single digits.
2. **Reduced bandwidth costs** — Only results are sent to the origin, not raw data.
3. **Higher availability** — Edge nodes operate independently; one region failing doesn't take down the whole service.
```

## Python Cell 2: Calling the REST API Directly (What the Worker Does)

Workers can't use the Python SDK — they call the REST API via `fetch`. This Python cell shows the exact same call using `httpx`, so you can see what's happening under the hood.

In [ ]:
import os

import httpx

ANTHROPIC_API_URL = "https://api.anthropic.com/v1/messages"
api_key = os.environ.get("ANTHROPIC_API_KEY")

# This is exactly what the Cloudflare Worker sends to Anthropic
headers = {
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01",
    "content-type": "application/json",
}

payload = {
    "model": "claude-haiku-4-5",
    "max_tokens": 256,
    "messages": [
        {"role": "user", "content": "In one sentence: why use Cloudflare Workers for AI APIs?"}
    ],
}

with httpx.Client() as client:
    response = client.post(ANTHROPIC_API_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()

# The Worker extracts data["content"][0]["text"] and returns it
reply = data["content"][0]["text"]
print(f"Reply: {reply}")
print(f"\nTokens used — input: {data['usage']['input_tokens']}, output: {data['usage']['output_tokens']}")

**Expected output:**
```
Reply: Cloudflare Workers bring AI inference closer to users with near-zero cold starts,
global distribution across 300+ locations, and a generous free tier — all without managing servers.

Tokens used — input: 22, output: 38
```

## Adding Streaming Support

For chat interfaces, streaming lets users see words appear immediately instead of waiting for the full response.

Anthropic uses Server-Sent Events (SSE) for streaming. Workers handle this natively with `TransformStream`.

**Why streaming works well with Workers:**
- The 30s CPU limit counts *active CPU time*, not wall-clock time
- While Claude is generating tokens, the Worker is idle — the clock barely moves
- You can stream multi-minute responses within the free tier

```javascript
// src/worker.js — streaming variant

export default {
  async fetch(request, env) {
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: CORS_HEADERS });
    }

    const { messages } = await request.json();

    // Add stream: true to the request body
    const anthropicResponse = await fetch('https://api.anthropic.com/v1/messages', {
      method: 'POST',
      headers: {
        'x-api-key': env.ANTHROPIC_API_KEY,
        'anthropic-version': '2023-06-01',
        'content-type': 'application/json',
      },
      body: JSON.stringify({
        model: 'claude-haiku-4-5',
        max_tokens: 1024,
        messages,
        stream: true,  // <-- enables SSE streaming
      }),
    });

    // Pipe the SSE stream directly from Anthropic to the client
    // No buffering — tokens flow through the Worker as they're generated
    return new Response(anthropicResponse.body, {
      headers: {
        'Content-Type': 'text/event-stream',
        'Cache-Control': 'no-cache',
        'Connection': 'keep-alive',
        ...CORS_HEADERS,
      },
    });
  },
};
```

**Client-side consumption (JavaScript):**

```javascript
const response = await fetch('https://your-worker.workers.dev/', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ messages: [{ role: 'user', content: 'Tell me a story' }] }),
});

const reader = response.body.getReader();
const decoder = new TextDecoder();

while (true) {
  const { done, value } = await reader.read();
  if (done) break;
  const chunk = decoder.decode(value);
  // Each chunk is an SSE event like: data: {"type":"content_block_delta",...}
  console.log(chunk);
}
```

## Using KV for Session State

Workers are stateless — each request starts fresh. To maintain conversation history across requests, use **Workers KV**: a globally distributed key-value store.

**Setup:**

```bash
# Create a KV namespace
wrangler kv namespace create SESSIONS
# Copy the namespace ID from the output
```

Add the binding to `wrangler.toml`:

```toml
[[kv_namespaces]]
binding = "SESSIONS"          # How you access it in code: env.SESSIONS
id = "abc123..."              # The ID from wrangler output above
```

**Worker with session persistence:**

```javascript
// src/worker.js — with KV session storage

const MAX_HISTORY = 20;   // Keep last 20 messages per session
const SESSION_TTL = 3600; // Sessions expire after 1 hour (seconds)

export default {
  async fetch(request, env) {
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: CORS_HEADERS });
    }

    const { session_id, message } = await request.json();

    // Load existing conversation history from KV
    // KV key: sessions/<session_id>
    let history = [];
    if (session_id) {
      const stored = await env.SESSIONS.get(`sessions/${session_id}`);
      if (stored) {
        history = JSON.parse(stored);
      }
    }

    // Append the new user message
    history.push({ role: 'user', content: message });

    // Call Claude with the full history
    const anthropicResponse = await fetch('https://api.anthropic.com/v1/messages', {
      method: 'POST',
      headers: {
        'x-api-key': env.ANTHROPIC_API_KEY,
        'anthropic-version': '2023-06-01',
        'content-type': 'application/json',
      },
      body: JSON.stringify({
        model: 'claude-haiku-4-5',
        max_tokens: 1024,
        messages: history,
      }),
    });

    const data = await anthropicResponse.json();
    const reply = data.content[0].text;

    // Append assistant reply and trim to MAX_HISTORY
    history.push({ role: 'assistant', content: reply });
    if (history.length > MAX_HISTORY) {
      history = history.slice(history.length - MAX_HISTORY);
    }

    // Save updated history back to KV with TTL
    const newSessionId = session_id || crypto.randomUUID();
    await env.SESSIONS.put(
      `sessions/${newSessionId}`,
      JSON.stringify(history),
      { expirationTtl: SESSION_TTL }  // Auto-cleanup after 1 hour
    );

    return new Response(
      JSON.stringify({ reply, session_id: newSessionId }),
      { headers: { 'Content-Type': 'application/json', ...CORS_HEADERS } }
    );
  },
};
```

**KV characteristics to know:**
- Reads are fast (cached at edge), writes propagate globally in ~60 seconds
- Free tier: 100K reads/day, 1K writes/day
- `expirationTtl` handles cleanup automatically — no cron job needed
- For high-frequency writes (real-time updates), use **Durable Objects** instead

## Testing Locally with Wrangler

Wrangler runs a local Worker runtime that closely mirrors production:

```bash
# Start local dev server (hot-reloads on file changes)
wrangler dev
# Worker is now available at http://localhost:8787
```

**Test with curl:**

```bash
# Basic chat
curl -X POST http://localhost:8787 \
  -H 'Content-Type: application/json' \
  -d '{"messages": [{"role": "user", "content": "Hello!"}]}'

# With session ID (for KV-enabled worker)
curl -X POST http://localhost:8787 \
  -H 'Content-Type: application/json' \
  -d '{"session_id": "test-123", "message": "What did I just say?"}'
```

**Local KV in dev:**
Wrangler persists KV data locally in `.wrangler/state/` during development — no reads/writes hit the real KV namespace. Your dev quota is safe.

**Tail logs in production:**
```bash
wrangler tail  # Stream live logs from your deployed Worker
```

## Deploying to Production

```bash
# Deploy to Cloudflare's global network
wrangler deploy
# Your Worker is instantly live at:
# https://claude-chat-worker.<your-subdomain>.workers.dev
```

**Add a custom domain:**
1. In the Cloudflare dashboard, go to Workers & Pages → your Worker → Settings → Triggers
2. Click "Add Custom Domain" and enter `api.yourdomain.com`
3. Cloudflare automatically provisions a TLS certificate

Or via wrangler.toml:
```toml
[triggers]
routes = [{ pattern = "api.yourdomain.com/chat", zone_name = "yourdomain.com" }]
```

**Rate limiting (built-in, no code needed):**
In the Cloudflare dashboard → Security → WAF → Rate Limiting Rules:
- Match: `http.request.uri.path eq "/"` and request method `POST`
- Rate: 60 requests per minute per IP
- Action: Block or challenge

This protects your Anthropic API budget from abuse at zero extra cost.

## Cost Analysis

### Cloudflare Workers costs

| Plan | Requests | CPU time | Price |
|------|----------|----------|-------|
| Free | 100K/day | 10ms/req | $0 |
| Paid | 10M/month | 30ms/req | $5/month |
| Additional | per million | per million ms | $0.30 / $0.02 |

For a chat API, each Worker invocation uses ~2-5ms of CPU (the rest is waiting for Anthropic). The free tier easily handles 100K conversations/day.

### Claude API costs (claude-haiku-4-5)

| Usage | Cost |
|-------|------|
| Input tokens | $0.80 per million |
| Output tokens | $4.00 per million |
| Typical chat turn (500 in / 200 out) | ~$0.0002 |
| 10,000 conversations/day | ~$2/day |

### Platform comparison

| Platform | Cold start | Global | Ops burden | Free tier |
|----------|-----------|--------|------------|----------|
| Cloudflare Workers | ~0ms | Yes (300+ locations) | Zero | 100K req/day |
| AWS Lambda | 100-500ms | No (one region) | Low | 1M req/month |
| Cloud Run | 1-5s | No (one region) | Medium | Limited |
| EC2 / VPS | N/A | No | High | None |

Workers wins on cold start and global distribution. Lambda wins on ecosystem (more triggers, more runtimes). For Claude-powered chat APIs, Workers is the clear choice.

## Full Production Worker

This combines all patterns: CORS, streaming, KV sessions, and error handling. Copy-paste into `src/worker.js`:

```javascript
/**
 * Claude Chat Worker — Production Ready
 * Supports: CORS, streaming, KV session persistence, error handling
 *
 * Required secrets (set with wrangler secret put):
 *   ANTHROPIC_API_KEY
 *
 * Required KV bindings (in wrangler.toml):
 *   [[kv_namespaces]]
 *   binding = "SESSIONS"
 *   id = "<your-namespace-id>"
 */

const ANTHROPIC_API_URL = 'https://api.anthropic.com/v1/messages';
const MODEL = 'claude-haiku-4-5';
const MAX_TOKENS = 1024;
const MAX_HISTORY = 20;
const SESSION_TTL = 3600;

const CORS_HEADERS = {
  'Access-Control-Allow-Origin': '*',
  'Access-Control-Allow-Methods': 'POST, OPTIONS',
  'Access-Control-Allow-Headers': 'Content-Type',
};

function jsonResponse(data, status = 200) {
  return new Response(JSON.stringify(data), {
    status,
    headers: { 'Content-Type': 'application/json', ...CORS_HEADERS },
  });
}

async function loadHistory(env, sessionId) {
  if (!sessionId) return [];
  const stored = await env.SESSIONS.get(`sessions/${sessionId}`);
  return stored ? JSON.parse(stored) : [];
}

async function saveHistory(env, sessionId, history) {
  const trimmed = history.length > MAX_HISTORY
    ? history.slice(history.length - MAX_HISTORY)
    : history;
  await env.SESSIONS.put(
    `sessions/${sessionId}`,
    JSON.stringify(trimmed),
    { expirationTtl: SESSION_TTL }
  );
  return trimmed;
}

export default {
  async fetch(request, env) {
    // CORS preflight
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: CORS_HEADERS });
    }

    if (request.method !== 'POST') {
      return jsonResponse({ error: 'Method not allowed' }, 405);
    }

    // Parse and validate request
    let body;
    try {
      body = await request.json();
    } catch {
      return jsonResponse({ error: 'Invalid JSON body' }, 400);
    }

    const { message, session_id, stream = false } = body;

    if (!message || typeof message !== 'string') {
      return jsonResponse({ error: 'message (string) is required' }, 400);
    }

    // Load history and append new message
    const history = await loadHistory(env, session_id);
    history.push({ role: 'user', content: message });

    // Call Anthropic
    let anthropicResponse;
    try {
      anthropicResponse = await fetch(ANTHROPIC_API_URL, {
        method: 'POST',
        headers: {
          'x-api-key': env.ANTHROPIC_API_KEY,
          'anthropic-version': '2023-06-01',
          'content-type': 'application/json',
        },
        body: JSON.stringify({
          model: MODEL,
          max_tokens: MAX_TOKENS,
          messages: history,
          stream,
        }),
      });
    } catch (err) {
      return jsonResponse({ error: 'Failed to reach Anthropic API', detail: err.message }, 502);
    }

    if (!anthropicResponse.ok) {
      const error = await anthropicResponse.json().catch(() => ({}));
      return jsonResponse({ error: 'Anthropic API error', detail: error }, anthropicResponse.status);
    }

    // Streaming: pipe SSE events directly to client
    if (stream) {
      return new Response(anthropicResponse.body, {
        headers: {
          'Content-Type': 'text/event-stream',
          'Cache-Control': 'no-cache',
          'Connection': 'keep-alive',
          ...CORS_HEADERS,
        },
      });
    }

    // Non-streaming: extract reply, save history, return JSON
    const data = await anthropicResponse.json();
    const reply = data.content[0].text;

    history.push({ role: 'assistant', content: reply });
    const newSessionId = session_id || crypto.randomUUID();
    await saveHistory(env, newSessionId, history);

    return jsonResponse({
      reply,
      session_id: newSessionId,
      usage: data.usage,
    });
  },
};
```

## Next Steps

Once your basic chat Worker is running, here are natural extensions:

### R2 for File Uploads
Cloudflare R2 is S3-compatible object storage with zero egress fees. Use it to accept file uploads and pass their content to Claude:
```javascript
// Upload a file, get a key, then reference it in your message
await env.UPLOADS.put(fileKey, fileBuffer);
const fileContent = await env.UPLOADS.get(fileKey).then(r => r.text());
```

### Durable Objects for WebSocket Connections
KV is eventually consistent and has write limits. For real-time chat applications (WebSockets), use **Durable Objects** — strongly consistent, stateful objects that live at a single location:
```javascript
// Each chat room gets its own Durable Object
const room = env.CHAT_ROOMS.get(env.CHAT_ROOMS.idFromName(roomId));
return room.fetch(request); // The DO handles WebSocket upgrades
```

### D1 for Structured Data
Cloudflare D1 is a serverless SQLite database. Use it to store user accounts, message history with full-text search, or analytics:
```javascript
await env.DB.prepare(
  'INSERT INTO messages (session_id, role, content, created_at) VALUES (?, ?, ?, ?)'
).bind(sessionId, 'user', message, Date.now()).run();
```

### System Prompts and Personas
Add a `system` field to the Anthropic request body to give Claude a persona or task-specific instructions:
```javascript
body: JSON.stringify({
  model: MODEL,
  max_tokens: MAX_TOKENS,
  system: 'You are a helpful customer support agent for Acme Corp. Be concise and friendly.',
  messages: history,
})
```

---

**References:**
- [Cloudflare Workers docs](https://developers.cloudflare.com/workers/)
- [Anthropic API reference](https://docs.anthropic.com/en/api/getting-started)
- [Workers KV docs](https://developers.cloudflare.com/kv/)
- [Wrangler CLI reference](https://developers.cloudflare.com/workers/wrangler/)